# Why does exact-Hessian collocation depend on five SLSQP steps?

This standalone A100 diagnostic separates two effects that the convergence benchmark previously bundled together:

1. **parameter initialization**: model defaults versus the parameters after five CPU SLSQP iterations; and
2. **boundary-state initialization**: a dynamically feasible rollout, data-informed states, or Twin4Build's automatic choice.

The six-arm factorial test is important because the earlier unseeded control explicitly forced `boundary_state_init="rollout"`. That is a useful parameter-only control, but it bypasses the production `"auto"` policy, which is designed to select data-informed states when the initial rollout is far from the measurements.

All CUDA arms use the same corrected scaling-and-squaring exponential, exact Hessian, and direct CUDA Graph replay. Each arm runs in a fresh subprocess. The notebook reports both the requested and resolved state initialization, the automatically measured warm-start fit, convergence, feasibility, rollout quality, and timing.

Local gradient methods are **not** generally expected to find the same optimum. Such a guarantee requires convexity and equivalent feasible sets. This model has nonlinear dynamics, bilinear terms, parameter bounds, free collocation boundary states, and a non-convex least-squares landscape; SLSQP's reduced single-shooting path and IPOPT's full-space constrained path can enter different basins.

In [ ]:
import json
import os
from pathlib import Path
import platform
import subprocess
import sys

import pandas as pd
import torch

REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
CANDIDATE_REF = "feature/issue-128/collocation-initialization"
ROOT = Path("/content/twin4build_collocation_initialization")
CHECKOUT = ROOT / "candidate"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIAGNOSTIC_HOURS = 120 if DEVICE == "cuda" else 24
DIAGNOSTIC_MAXITER = 1000 if DEVICE == "cuda" else 20
if DEVICE == "cpu":
    print("CUDA unavailable: running the 24-hour/20-iteration CPU screen.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{CANDIDATE_REF}",
    ],
    check=True,
)

ROOT.mkdir(parents=True, exist_ok=True)
if CHECKOUT.exists():
    subprocess.run(
        ["git", "fetch", "--quiet", "origin", CANDIDATE_REF],
        cwd=CHECKOUT,
        check=True,
    )
    subprocess.run(
        ["git", "reset", "--hard", f"origin/{CANDIDATE_REF}"],
        cwd=CHECKOUT,
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--quiet",
            "--depth",
            "1",
            "--branch",
            CANDIDATE_REF,
            REPO_URL,
            str(CHECKOUT),
        ],
        check=True,
    )

transcription_source = (
    CHECKOUT / "twin4build/estimator/_transcription.py"
).read_text(encoding="utf-8")
if "class _CudaGraphCallable" not in transcription_source:
    raise RuntimeError("Candidate checkout does not contain direct Hessian capture.")

props = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
HARDWARE = {
    "cpu": platform.processor() or platform.machine(),
    "cpu_logical": os.cpu_count(),
    "device": DEVICE,
    "gpu": None if props is None else props.name,
    "gpu_memory_gb": None if props is None else props.total_memory / 1e9,
    "diagnostic_hours": DIAGNOSTIC_HOURS,
    "diagnostic_maxiter": DIAGNOSTIC_MAXITER,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "candidate_ref": CANDIDATE_REF,
}
print(json.dumps(HARDWARE, indent=2))

In [ ]:
# Reuse the already validated five-day model/measurement builder from the
# convergence benchmark, then make only the arm logic specific to this study.
source_notebook = json.loads(
    (
        CHECKOUT
        / "twin4build/examples/cpu_slsqp_vs_cuda_exact_collocation_benchmark.ipynb"
    ).read_text(encoding="utf-8")
)
runner_cell = next(
    cell
    for cell in source_notebook["cells"]
    if cell.get("cell_type") == "code"
    and "RUNNER.write_text" in "".join(cell.get("source", []))
)
exec(compile("".join(runner_cell["source"]), "validated_runner_cell", "exec"))

text = RUNNER.read_text(encoding="utf-8")


def replace_once(old, new):
    global text
    if text.count(old) != 1:
        raise RuntimeError(
            f"Expected exactly one validated-runner fragment, found {text.count(old)}: {old[:80]!r}"
        )
    text = text.replace(old, new)


arms = [
    "defaults_rollout",
    "defaults_data",
    "defaults_auto",
    "seed5_rollout",
    "seed5_data",
    "seed5_auto",
]

replace_once(
    'END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]',
    'END = [START[0] + datetime.timedelta(hours=int(os.environ.get("T4B_DIAG_HOURS", "120")))]',
)
replace_once(
    'COLLOCATION_ITERS = 1000',
    'COLLOCATION_ITERS = int(os.environ.get("T4B_DIAG_MAXITER", "1000"))',
)
replace_once(
    'device = "cpu" if arm == "cpu_slsqp" else "cuda"',
    'device = os.environ.get("T4B_DIAG_DEVICE", "cuda")',
)
replace_once(
    '            "capture_hessian": cuda_capture[arm],',
    '            "capture_hessian": cuda_capture[arm] and device == "cuda",',
)
replace_once(
    '    "hours": 120,',
    '    "hours": int(os.environ.get("T4B_DIAG_HOURS", "120")),',
)

replace_once(
    '''cuda_capture = {
    "cuda_fixed_eager": False,
    "cuda_direct_graph": True,
    "cuda_direct_graph_unseeded": True,
}''',
    "cuda_capture = " + repr({arm: True for arm in arms}),
)
replace_once(
    '''if arm != "cpu_slsqp" and arm not in cuda_capture:
    raise ValueError(f"Unknown arm {arm!r}")''',
    '''if arm not in cuda_capture:
    raise ValueError(f"Unknown arm {arm!r}")
init_mode = arm.rsplit("_", 1)[1]''',
)
replace_once(
    '    parameters = build_parameters(model)\n    started = time.perf_counter()',
    '    parameters = build_parameters(model)\n    default_values = read_parameter_values(parameters)\n    started = time.perf_counter()',
)
replace_once(
    '        "seconds": seed_seconds,',
    '        "seconds": seed_seconds,\n        "hours": int(os.environ.get("T4B_DIAG_HOURS", "120")),',
)
replace_once(
    '        "values": read_parameter_values(parameters),',
    '''        "values": read_parameter_values(parameters),
        "default_values": default_values,
        "parameter_names": [
            "/".join(c.id for c in (entry[0] if isinstance(entry[0], list) else [entry[0]]))
            + "." + entry[1]
            for entry in parameters
        ],
        "bounds": [[entry[3], entry[4]] for entry in parameters],''',
)
replace_once(
    'uses_shared_seed = arm != "cuda_direct_graph_unseeded"',
    'uses_shared_seed = arm.startswith("seed5_")',
)
replace_once(
    '            "boundary_state_init": "rollout",',
    '            "boundary_state_init": init_mode,',
)
replace_once(
    '    "parameter_seed": "cpu_slsqp_5" if uses_shared_seed else "model_defaults",',
    '''    "parameter_seed": "cpu_slsqp_5" if uses_shared_seed else "model_defaults",
    "requested_boundary_state_init": init_mode,
    "resolved_boundary_state_init": audit.get("boundary_state_init"),
    "warm_start_fit": audit.get("warm_start_fit"),''',
)

RUNNER.write_text(text, encoding="utf-8")
print(f"Wrote initialization runner with arms: {arms}")

In [ ]:
SEED_FILE = ROOT / "shared_cpu_seed.json"
RESULT_FILES = {arm: ROOT / f"{arm}_convergence.json" for arm in arms}
CURRENT_REF = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=CHECKOUT, text=True
).strip()


def run_process(arm, out_file):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(CHECKOUT) + os.pathsep + env.get("PYTHONPATH", "")
    env["T4B_DIAG_DEVICE"] = DEVICE
    env["T4B_DIAG_HOURS"] = str(DIAGNOSTIC_HOURS)
    env["T4B_DIAG_MAXITER"] = str(DIAGNOSTIC_MAXITER)
    completed = subprocess.run(
        [sys.executable, str(RUNNER), str(CHECKOUT), arm, str(SEED_FILE), str(out_file)],
        cwd=ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode:
        raise RuntimeError(f"{arm} failed with exit code {completed.returncode}")


def result_is_current(path, arm=None):
    if not path.exists():
        return False
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        common = (
            payload.get("ref") == CURRENT_REF
            and payload.get("hours") == DIAGNOSTIC_HOURS
        )
        if arm is None:
            return common
        return (
            common
            and payload.get("arm") == arm
            and payload.get("device") == DEVICE
            and payload.get("iteration_limit") == DIAGNOSTIC_MAXITER
        )
    except Exception:
        return False


if not result_is_current(SEED_FILE):
    print("Running shared five-iteration CPU SLSQP seed ...", flush=True)
    run_process("seed", ROOT / "unused_seed_result.json")
else:
    print("Reusing current five-iteration seed.")

for arm, result_file in RESULT_FILES.items():
    if result_is_current(result_file, arm):
        print(f"Reusing {arm}.")
        continue
    print(f"Running {arm} ...", flush=True)
    run_process(arm, result_file)

rows = [json.loads(path.read_text(encoding="utf-8")) for path in RESULT_FILES.values()]
print("All six initialization arms completed.")

In [ ]:
summary = pd.DataFrame(rows)
columns = [
    "arm",
    "parameter_seed",
    "requested_boundary_state_init",
    "resolved_boundary_state_init",
    "warm_start_fit",
    "iterations",
    "success",
    "message",
    "final_objective",
    "max_defect",
    "pooled_rollout_objective",
    "estimate_seconds",
    "peak_cuda_memory_gb",
    "rmse_temperature",
    "rmse_valve_position",
    "rmse_damper_position",
    "rmse_co2",
]
display(summary[columns])

seed = json.loads(SEED_FILE.read_text(encoding="utf-8"))
movement = pd.DataFrame(
    {
        "parameter": seed["parameter_names"],
        "default": seed["default_values"],
        "slsqp5": seed["values"],
        "lower": [bound[0] for bound in seed["bounds"]],
        "upper": [bound[1] for bound in seed["bounds"]],
    }
)
movement["normalized_delta"] = (
    (movement.slsqp5 - movement.default) / (movement.upper - movement.lower)
)
movement["absolute_normalized_delta"] = movement.normalized_delta.abs()
print("Five-step SLSQP parameter movement, largest first:")
display(movement.sort_values("absolute_normalized_delta", ascending=False))

comparison = summary.pivot(
    index="requested_boundary_state_init",
    columns="parameter_seed",
    values=[
        "success",
        "iterations",
        "final_objective",
        "max_defect",
        "pooled_rollout_objective",
        "warm_start_fit",
    ],
)
display(comparison)

def row(name):
    return summary.loc[summary.arm == name].iloc[0]

cold_rollout = row("defaults_rollout")
cold_data = row("defaults_data")
cold_auto = row("defaults_auto")
seeded_rollout = row("seed5_rollout")

print("\nInterpretation gates:")
print(
    f"- defaults/auto resolved to {cold_auto.resolved_boundary_state_init!r} "
    f"from warm_start_fit={cold_auto.warm_start_fit:.6g}."
)
if bool(cold_auto.success) and not bool(cold_rollout.success):
    print(
        "- The production auto policy removes the apparent need for SLSQP; "
        "the earlier failure was specific to forcing a poor rollout-state start."
    )
elif bool(cold_data.success) and not bool(cold_rollout.success):
    print(
        "- Data-informed states rescue default parameters, but the auto threshold "
        "or its resolved choice needs adjustment."
    )
elif not bool(cold_data.success) and bool(seeded_rollout.success):
    print(
        "- State initialization alone is insufficient: the five-step parameter "
        "movement crosses into a materially better basin. Run the seed-budget "
        "and interpolation follow-up from issue #128."
    )
else:
    print(
        "- No single factor explains the outcome. Inspect the 2x3 interaction, "
        "then run perturbation starts around the separating parameter points."
    )

for candidate in (cold_rollout, cold_data, cold_auto, seeded_rollout):
    if candidate.success and candidate.max_defect > 1e-6:
        print(f"WARNING: reject {candidate.arm}: reported success but defect exceeds 1e-6")